# Jersey colour classification in football video


In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent.resolve()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "dataset_v1"
REPORT_DIR = PROJECT_ROOT / "reports"

print("cwd =", Path.cwd())

In [ ]:
import torch
import pandas as pd
import numpy as np

from itertools import product
from IPython.display import display

from src.dataset import load_manifest
from src.extract_embeddings import (
    get_embeddings,
    extract_all_models,
    extract_all_finetuned,
)
from src.classification_clustering import (
    compare_models,
    evaluate_single_method,
    run_clustering,
)
from scripts.benchmark import (
    run_benchmark,
    summarize_benchmark,
    remove_game_from_benchmark,
)

In [ ]:
import warnings
warnings.filterwarnings("ignore", module='umap')

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
manifest_path = DATA_DIR / "manifest_with_splits.csv"

df = load_manifest(manifest_path)

print("dataset size:", len(df))
df.head()

In [ ]:
games   = ['game_30', 'game_39', 'game_44', 'game_46']
methods = ["kmeans", "hdbscan", "gmm"]
configs = [
    ("raw",      False, False, False),
    ("umap",     True,  False, False),
    ("umap_pca", True,  True,  False),
]

pretrained_names = ["osnet", "dino"]

finetuned_configs = {
    # "osnet_triplet": ("osnet", "checkpoints/osnet_triplet_best.pth"),
    "osnet_supcon":  ("osnet", "checkpoints/osnet_supcon_best.pth"),
    "dino_supcon":   ("dino",  "checkpoints/dino_supcon_best.pth"),
    # "dino_triplet":  ("dino",  "checkpoints/dino_triplet_best.pth"),
}

all_model_names = pretrained_names + list(finetuned_configs.keys())
total = len(games) * len(all_model_names) * len(methods) * len(configs)
step  = 0
rows  = []


In [ ]:
benchmark_df = run_benchmark(
    games            = games,
    df               = df,
    all_model_names  = all_model_names,
    pretrained_names = pretrained_names,
    finetuned_configs= finetuned_configs,
    methods          = methods,
    configs          = configs,
    device           = device,
    csv_path         = REPORT_DIR / "benchmark.csv",
)


In [ ]:
pd.set_option('display.max_rows', 200)  

benchmark_df

In [ ]:
summarize_benchmark(benchmark_df)

In [ ]:
mask = (
    (benchmark_df["model"] == "osnet_supcon") &
    (benchmark_df["method"].isin(["gmm", "kmeans"])) &
    (benchmark_df["config"] == "umap")
)

display(benchmark_df[mask][[
    "game", "model", "method", "config",
    "clustering_accuracy", "macro_f1_cluster", "noise_fraction"
]].sort_values(["method", "game"]).round(4))